# YOLOv26 Optimization Pipeline

This notebook demonstrates the complete optimization pipeline for deploying YOLOv26 on ESP-DL. It utilizes Post-Training Quantization (PTQ), Trained Quantization Thresholds (TQT) for accuracy recovery, and direct LUT Fusion for hardware-accelerated INT16 activations.

In [ ]:
# ==========================================
# CELL 1: Standard Imports & Paths
# ==========================================
import os
import sys

# Add local directories to path to ensure standalone execution
sys.path.append('scripts')

import torch
import types
from esp_ppq.api import get_target_platform
import esp_ppq.lib as PFL
from esp_ppq.executor import TorchExecutor
from esp_ppq.core import QuantizationVisibility, TargetPlatform
from esp_ppq.api.interface import load_onnx_graph
from esp_ppq.quantization.optim import (
    QuantizeSimplifyPass, QuantizeFusionPass, ParameterQuantizePass,
    RuntimeCalibrationPass, PassiveParameterQuantizePass, QuantAlignmentPass,
    TrainedQuantizationThresholdPass
)

In [ ]:
# ==========================================
# CELL 2: User Configurations
# ==========================================
# --- Important Configuration ---
IMG_SZ_I = 512
PLATFORM = "p4"  
DATA_YAML_FILE_I = "coco.yaml" 
INT16_LUT_STEP_I = 32

In [ ]:
# ==========================================
# CELL 3: Configuration Injection
# ==========================================
class QATConfig:
    IMG_SZ = IMG_SZ_I
    DEVICE = "cuda" if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu"
    DATA_YAML_FILE = DATA_YAML_FILE_I
    BATCH_SIZE = 24           # Defines the batch size during the Calibration & TQT process
    CALIB_MAX_IMAGES = 1024 #2560  # The total subset of images passed to TQT for threshold refinement
    CALIB_VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    DATA_FALLBACK_PATH = "coco2017/images/train2017"
    CALIB_STEPS = 8
    QUANT_CALIB_METHOD = "kl" #"percentile" 
    QUANT_ALIGNMENT = "Align to Output"
    TARGET_PLATFORM = get_target_platform("esp32" + PLATFORM, 8)
    
    # LUT Hyperparameters
    INT16_LUT_STEP = INT16_LUT_STEP_I
    
    # TQT Specific Hyperparameters for Accuracy Recovery
    TQT_STEPS = 64
    TQT_LR = 2e-5
    TQT_INT_LAMBDA = 0.25
    TQT_BLOCK_SIZE = 10
    TQT_COLLECTING_DEVICE = "cpu"
    
    BASE_DIR = os.getcwd()
    MODEL_NAME = "yolo26n"
    PT_FILE = f"{MODEL_NAME}.pt"
    ONNX_FILE = f"{MODEL_NAME}_export.onnx"
    
    ESPDL_OUTPUT_DIR = os.path.join(BASE_DIR, "output", f"{DATA_YAML_FILE_I[:-5]}_{IMG_SZ_I}_s8_{PLATFORM}")
    ONNX_PATH = os.path.join(ESPDL_OUTPUT_DIR, ONNX_FILE)

# CRITICAL: Inject into sys.modules BEFORE importing local scripts that rely on config.QATConfig
if 'config' not in sys.modules:
    sys.modules['config'] = types.ModuleType('config')
sys.modules['config'].QATConfig = QATConfig

In [ ]:
# ==========================================
# CELL 4: Local Modules & Environment Setup
# ==========================================
# Local Sub-Module Imports (Now safe because 'config' exists)
from utils import seed_everything, register_mod_op, get_exclusive_ancestors
from dataset import get_calibration_loader
from ultralytics.data.utils import check_det_dataset
from esp_ppq_patch import apply_esp_ppq_patches
from esp_ppq_patch_2 import apply_addlut_patch
from notebook_helpers import extract_model_meta, prepare_onnx, prune_graph_safely

# Advanced LUT Imports natively located in esp_ppq_lut/
from esp_ppq_lut.passes import EspdlLUTFusionPass
from esp_ppq_lut.exporter import HardwareAwareEspdlExporter

os.makedirs(QATConfig.ESPDL_OUTPUT_DIR, exist_ok=True)

# Setup Native esp_ppq_lut environment
import esp_ppq_lut as esp_lut
esp_lut.initialize(step=QATConfig.INT16_LUT_STEP, verbose=True)

seed_everything(1234)
register_mod_op()
apply_esp_ppq_patches()
apply_addlut_patch()

print("Environment and Configuration setup complete.")

In [ ]:
# ==========================================
# CELL 5: ONNX Export & Metadata Extraction
# ==========================================
prepare_onnx()
model_meta = extract_model_meta()

In [ ]:
# apply patches (injected)
import sys
sys.path.insert(0, r"C:\Users\orani\bilel\git_projects\TQT\low-level_optimizations\esp-ppq")
import custom_ops_patch

In [ ]:
# ==========================================
# CELL 6: Quantizer Initialization & Branch Separation
# ==========================================
print("Loading ONNX Graph into ESP-PPQ...")
graph = load_onnx_graph(onnx_import_file=QATConfig.ONNX_PATH)



################### load native model injection
from esp_ppq.api.interface import load_native_graph
n_path = r"C:\Users\orani\bilel\git_projects\TQT\neural-morphing\integrations\yolo26n_v2\Transformation1_SiLU_to_HardSiLU\output\T1_silu_hsilu_512_p4\morphed_hsilu.native"
graph = load_native_graph(n_path)
###################


output_names = list(graph.outputs.keys())
aux_ops = set()
main_ops = set()

# Separate Aux (training) branch from Main (inference) branch
if len(output_names) >= 6:
    aux_outputs = output_names[0:3]
    main_outputs = output_names[3:6]
    aux_ops = get_exclusive_ancestors(graph, aux_outputs, main_outputs)
    main_ops = get_exclusive_ancestors(graph, main_outputs, aux_outputs)

quantizer = PFL.Quantizer(platform=QATConfig.TARGET_PLATFORM, graph=graph)
dispatching_table = PFL.Dispatcher(graph=graph, method="conservative").dispatch(
    quantizer.quant_operation_types
)

for opname, platform in dispatching_table.items():
    if platform == TargetPlatform.UNSPECIFIED:
        dispatching_table[opname] = TargetPlatform(quantizer.target_platform)

# Force Aux Branch to FP32 (We don't care about it, it will be pruned later)
for op in aux_ops:
    if op.name in dispatching_table:
        dispatching_table[op.name] = TargetPlatform.FP32

In [ ]:
# ==========================================
# CELL 7: Hardware Precision Targets (INT16 / FP32)
# ==========================================
INT16_PLATFORM = get_target_platform("esp32" + PLATFORM, 16)

# Force high-sensitivity Head and Neck exit layers to INT16
int16_layers = {
    # Neck Exits
    "/model.16/cv2/conv/Conv", "/model.16/cv2/conv/Conv/Swish",
    "/model.19/cv2/conv/Conv", "/model.19/cv2/conv/Conv/Swish",
    "/model.22/cv2/conv/Conv", "/model.22/cv2/conv/Conv/Swish",
    # One2One Box Head
    "/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv", "/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv", "/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.0/one2one_cv2.0.2/Conv",
    "/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv", "/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv", "/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.1/one2one_cv2.1.2/Conv",
    "/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv", "/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv", "/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.2/one2one_cv2.2.2/Conv",
    # One2One Class Head
    "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.2/Conv",
    "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.2/Conv",
    "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.2/Conv"
}

for op in graph.operations.values():
    if op.name in dispatching_table and op.name in int16_layers:
        dispatching_table[op.name] = INT16_PLATFORM

# Force Concat nodes to FP32 (Workaround for ESP-DL alignment limit)
fp32_layers = {"/model.23/Concat_5", "/model.23/Concat_3", "/model.23/Concat_4"}
for op in main_ops:
    if op.name in fp32_layers:
        dispatching_table[op.name] = TargetPlatform.FP32

print("Applying Dispatcher Types...")
for op in graph.operations.values():
    quantizer.quantize_operation(op_name=op.name, platform=dispatching_table[op.name])

In [ ]:
# ==========================================
# CELL 8: Linear Optimization Pipeline (PTQ + TQT + LUT)
# ==========================================
print("Running Linear Optimization Pipeline (Calibration -> TQT -> LUT)...")
data_cfg = check_det_dataset(QATConfig.DATA_YAML_FILE)
cali_loader = get_calibration_loader(data_cfg)

executor = TorchExecutor(graph=graph)
dummy_input = torch.zeros([1, 3, QATConfig.IMG_SZ, QATConfig.IMG_SZ]).to(QATConfig.DEVICE)
executor.tracing_operation_meta(inputs=dummy_input)

pipeline = PFL.Pipeline([
    QuantizeSimplifyPass(),
    QuantizeFusionPass(activation_type=quantizer.activation_fusion_types),
    ParameterQuantizePass(),
    RuntimeCalibrationPass(method=QATConfig.QUANT_CALIB_METHOD),
    
    # Accuracy Recovery
    #TrainedQuantizationThresholdPass(
    #    steps=QATConfig.TQT_STEPS,                         
    #    lr=QATConfig.TQT_LR,                           
    #    int_lambda=QATConfig.TQT_INT_LAMBDA,
    #    block_size=QATConfig.TQT_BLOCK_SIZE,
    #    collecting_device=QATConfig.TQT_COLLECTING_DEVICE
    #), 

    PassiveParameterQuantizePass(clip_visiblity=QuantizationVisibility.EXPORT_WHEN_ACTIVE),
    QuantAlignmentPass(elementwise_alignment=QATConfig.QUANT_ALIGNMENT),
    
    # Direct LUT Conversion for HW int16 Swish emulation
    EspdlLUTFusionPass(
        target_ops=['Swish'],
        lut_step=QATConfig.INT16_LUT_STEP
    ) 
])

pipeline.optimize(
    calib_steps=QATConfig.CALIB_STEPS,
    collate_fn=(lambda x: x.type(torch.float).to(QATConfig.DEVICE)),
    graph=graph,
    dataloader=cali_loader,
    executor=executor,
)
print("Pipeline complete.")

In [ ]:
# ==========================================
# CELL 9: Validation (Check Quantized Graph mAP)
# ==========================================
"""
from trainer import QATTrainer # Used ONLY as a wrapper to call the validator
print("Evaluating Target ESP-DL Emulated mAP...")
dummy_trainer = QATTrainer(graph=graph, model_meta=model_meta, device=QATConfig.DEVICE)
val_mAP = dummy_trainer.eval()
print(f"Final Quantized mAP50-95: {val_mAP:.3f}")
"""


In [ ]:
# 0.35
# 0.348

In [ ]:
list(graph.outputs.keys())

In [ ]:
# ==========================================
# CELL 10: Graph Surgery (Pruning and Output Tearing)
# ==========================================
print("Slicing output Concat nodes into 6 discrete tensors...")

# 1. Remove Aux Heads
output_names = list(graph.outputs.keys())
if len(output_names) >= 6:
    for name in output_names[0:3]:
        if name in graph.outputs: graph.outputs.pop(name)
    prune_graph_safely(graph)

In [ ]:
prune_graph_safely(graph)

In [ ]:


# 2. Slice the Concat into Box/Cls
targets = ["one2one_p3", "one2one_p4", "one2one_p5"]
collected_outputs = {}
for target_name in targets:
    if target_name in graph.outputs:
        print(target_name)
        original_output_var = graph.variables[target_name]
        producer = original_output_var.source_op

        if producer and producer.type == "Concat":
            print("producer and producer.type == Concat")
            box_var, cls_var = None, None

            if model_meta['nc'] in producer.inputs[0].shape:
                cls_var = producer.inputs[0]
                box_var = producer.inputs[1]
            else:
                cls_var = producer.inputs[1]
                box_var = producer.inputs[0]


            if box_var and cls_var:
                print("box_var and cls_var")
                pair_config = [
                    (box_var, f"{target_name}_box"),
                    (cls_var, f"{target_name}_cls"),
                ]
                for var, new_name in pair_config:
                    old_name = var.name
                    if old_name in graph.variables: graph.variables.pop(old_name)
                    var._name = new_name
                    graph.variables[new_name] = var
                    collected_outputs[new_name] = var

                graph.outputs.pop(target_name)
                graph.remove_operation(producer, keep_coherence=False)
                for var in producer.inputs:
                    if producer in var.dest_ops: var.dest_ops.remove(producer)

# 3. Enforce precise output order matching ESPdl C++ expectations
final_output_list = [
    "one2one_p3_box", "one2one_p3_cls",
    "one2one_p4_box", "one2one_p4_cls",
    "one2one_p5_box", "one2one_p5_cls"
]
graph.outputs.clear()
for name in final_output_list:
    if name in collected_outputs:
        graph.outputs[name] = collected_outputs[name]

prune_graph_safely(graph)


In [ ]:
collected_outputs

In [ ]:
graph.outputs

In [ ]:
import math
for op in graph.operations.values():
    for cfg in (list(op.input_quant_config or []) + list(op.output_quant_config or [])):
        if cfg.scale is not None:
            s = cfg.scale.item()
            e = math.log2(s)
            if abs(e - round(e)) > 1e-10:
                print(f"NON-POW2: {op.name} scale={s:.10f} log2={e:.6f}")

In [ ]:
for op in graph.operations.values():
    if op.type == 'HardSiluPie8':
        in_cfg = op.input_quant_config[0] if op.input_quant_config else None
        out_cfg = op.output_quant_config[0] if op.output_quant_config else None
        # Check if input/output configs are the same object (shared = fused)
        print(f"{op.name}: in_scale={in_cfg.scale.item():.6f} out_scale={out_cfg.scale.item():.6f} "
              f"shared={in_cfg is out_cfg} in_state={in_cfg.state} out_state={out_cfg.state}")
        # Also check the Conv feeding into this HardSiluPie8
        src = op.inputs[0].source_op
        if src:
            src_out = src.output_quant_config[0]
            print(f"  ← Conv out_scale={src_out.scale.item():.6f} shared_with_hsilu_in={src_out is in_cfg}")
        break  # just first one

In [ ]:
import torch
from esp_ppq.executor import TorchExecutor
from esp_ppq_lut import set_simulation_mode, SimulationMode
from notebook_helpers import espdl_preprocess, enable_fp64_conv, disable_fp64_conv, _register_fp64_conv_handler
import cv2, numpy as np
# Preprocess
im0 = cv2.imread(os.path.join(QATConfig.BASE_DIR, "results", "bus.jpg"))
im = espdl_preprocess(im0, (512, 512))
im_chw = np.ascontiguousarray(im[..., ::-1].transpose(2, 0, 1))
test_input = torch.from_numpy(im_chw).float().div(255.0).unsqueeze(0)
# Inference
set_simulation_mode(SimulationMode.SIMULATION)
_register_fp64_conv_handler()
enable_fp64_conv()
executor = TorchExecutor(graph=graph)
raw_outputs = executor.forward(test_input)
disable_fp64_conv()
output_keys = list(graph.outputs.keys())
cls_var = graph.outputs["one2one_p4_cls"]
out_cfg = cls_var.source_op.output_quant_config[0]
raw = raw_outputs[output_keys.index("one2one_p4_cls")].detach().cpu().float()
int_vals = torch.round(raw / out_cfg.scale.item()).to(torch.int32)
flat = int_vals.flatten()
top_val, top_idx = flat.topk(5)
print(f"out_scale={out_cfg.scale.item()}")
print(f"Top-5 INT8 values: {top_val.tolist()}")
print(f"Top-5 indices: {top_idx.tolist()}")
print(f"Top-5 sigmoid: {torch.sigmoid(top_val.float() * out_cfg.scale.item()).tolist()}")

In [ ]:
import math
for op in graph.operations.values():
    if op.type != 'HardSiluPie8':
        continue
    out_cfg = op.output_quant_config[0]
    out_var = op.outputs[0]
    # Find the next op (consumer of this output)
    for next_op in out_var.dest_ops:
        if next_op.input_quant_config:
            # Find which input index connects to our output
            for idx, inp_var in enumerate(next_op.inputs):
                if inp_var.name == out_var.name:
                    next_in_cfg = next_op.input_quant_config[idx]
                    if next_in_cfg.scale is not None:
                        s1 = out_cfg.scale.item()
                        s2 = next_in_cfg.scale.item()
                        if abs(s1 - s2) > 1e-10:
                            print(f"MISMATCH: {op.name} out={s1} → {next_op.name} in={s2}")
                    break
   

In [ ]:
from esp_ppq.api import load_native_graph
from esp_ppq.parser import NativeExporter
# Save
exporter = NativeExporter()
exporter.export(file_path="yolo26n_model.native", graph=graph)

In [ ]:
"""
# Fix HardSiluPie8 → Add scale mismatches
fixed = 0
for op in graph.operations.values():
    if op.type != 'HardSiluPie8':
        continue
    out_var = op.outputs[0]
    for next_op in out_var.dest_ops:
        if not next_op.input_quant_config:
            continue
        for idx, inp in enumerate(next_op.inputs):
            if inp.name == out_var.name and idx < len(next_op.input_quant_config):
                next_cfg = next_op.input_quant_config[idx]
                out_cfg = op.output_quant_config[0]
                if next_cfg.scale is not None and abs(out_cfg.scale.item() - next_cfg.scale.item()) > 1e-10:
                    print(f"FIX: {op.name} out_scale {out_cfg.scale.item()} → {next_cfg.scale.item()}")
                    op.output_quant_config[0] = next_cfg  # share the downstream config
                    fixed += 1
print(f"Fixed {fixed} HardSiluPie8 scale mismatches")
"""

In [ ]:
# ==========================================
# CELL 10.1: Inference Preview (ESP-DL Emulated)
# ==========================================
# Runs the quantized graph on a test image using the same preprocessing
# as the ESP-DL C++ runtime (nearest-neighbour resize + letterbox padding,
# pad_val=114 — matching YOLO26::preprocess() in yolo26.cpp).
#
# Post-processing mirrors YOLO26::postprocess() + YOLO26::decode_grid()
# from esp-dl/models/yolo26/yolo26.cpp exactly:
#   - Sigmoid applied to raw cls scores
#   - Box decoded as: x1=(cx+0.5 - d_l)*stride, x2=(cx+0.5 + d_r)*stride
#   - YOLOv26 one2one head is NMS-FREE: top-K sort by score only
#
# Output filename follows the .espdl export convention:
#   <stem>_<img_sz>_s8_<platform>.jpg   e.g. bus_512_s8_p4.jpg
%matplotlib inline
from notebook_helpers import eval_espdl_model
import os

TEST_IMAGE = os.path.join(QATConfig.BASE_DIR, "results", "bus.jpg")
OUTPUT_DIR = os.path.join(QATConfig.BASE_DIR, "results")

predictions, saved_path = eval_espdl_model(
    test_image_path = TEST_IMAGE,
    graph           = graph,
    target_img_sz   = QATConfig.IMG_SZ,
    data_yaml       = QATConfig.DATA_YAML_FILE,
    platform        = PLATFORM,
    conf_thresh     = 0.25,
    output_dir      = OUTPUT_DIR,
)

print(f"\nDetections ({len(predictions)}):")
for p in predictions:
    print(f"  [{p['class_id']:3d}] {p['class']:20s}  conf={p['score']:.3f}  box={[round(v) for v in p['box']]}")


In [ ]:
# PATCH: Print all Transpose ops with MCU-physical input shapes + perm
def patch_print_transpose_cases():
    from esp_ppq.parser.espdl.espdl_typedef import ExporterPatternInfo
    from esp_ppq.parser.espdl.espdl_graph_utils import transpose_shape
    import esp_ppq.parser.espdl_exporter as _exp

    _orig_export_graph = _exp.EspdlExporter.export_graph

    def patched_export_graph(self, graph, **kwargs):
        info = ExporterPatternInfo()
        print("\n=== TRANSPOSE CASES ON MCU ===")
        for op in graph.topological_sort():
            if op.type == "Transpose":
                perm = list(op.attributes.get("perm", []))
                in_var = op.inputs[0]
                if in_var.is_parameter:
                    continue
                in_perm = info.get_var_permute(in_var.name)
                logical_shape = list(in_var.shape)
                if in_perm and logical_shape:
                    phys_input = transpose_shape(logical_shape, in_perm)
                else:
                    phys_input = logical_shape
                print(f"  {str(phys_input):25s} → T{perm}   [{op.name}]")
        print("=== END ===\n")

        return _orig_export_graph(self, graph, **kwargs)

    _exp.EspdlExporter.export_graph = patched_export_graph
    print("[PATCH] Transpose MCU-shape printer installed")

# patch_print_transpose_cases()


In [ ]:
# ==========================================
# CELL 11: Final Export
# ==========================================
final_espdl_path = os.path.join(QATConfig.ESPDL_OUTPUT_DIR, f"yolo26n_{QATConfig.IMG_SZ}_s8_{PLATFORM}.espdl")

# Using the Local Advanced Exporter that fixes the Pivot bug natively
exporter = PFL.Exporter(platform=QATConfig.TARGET_PLATFORM)
exporter.export(final_espdl_path, graph=graph, int16_lut_step=QATConfig.INT16_LUT_STEP)
print(f"Deployment Model exported seamlessly to {final_espdl_path}")

In [ ]:
# ==========================================
# DEBUG: Dump /model.10 subgraph AFTER layout transform
# ==========================================
from esp_ppq.parser.espdl.layout_patterns import reset_graph_layout
from esp_ppq.parser.espdl.espdl_typedef import ExporterPatternInfo

# PPQ's native graph copy (preserves Variable/Operation references)
debug_graph = graph.copy(copy_value=True)

info = ExporterPatternInfo()
info.reset()

# Apply the same layout transform the exporter does internally
reset_graph_layout(debug_graph)

print("=" * 80)
print("SUBGRAPH: /model.10 (after layout transform)")
print("=" * 80)

for op in debug_graph.topological_sort():
    if not op.name.startswith("/model.10"):
        continue

    print(f"\n{'─' * 60}")
    print(f"  OP: {op.name}")
    print(f"  Type: {op.type}")

    if op.attributes:
        for k, v in op.attributes.items():
            print(f"  Attr[{k}]: {v}")

    for i, var in enumerate(op.inputs):
        param_tag = " [PARAM]" if var.is_parameter else ""
        shape = list(var.shape) if var.shape is not None else "?"
        perm = info.get_var_permute(var.name)
        perm_tag = f"  perm={perm}" if perm else ""
        print(f"  IN[{i}]:  {var.name}  shape={shape}{param_tag}{perm_tag}")

    for i, var in enumerate(op.outputs):
        shape = list(var.shape) if var.shape is not None else "?"
        perm = info.get_var_permute(var.name)
        perm_tag = f"  perm={perm}" if perm else ""
        print(f"  OUT[{i}]: {var.name}  shape={shape}{perm_tag}")

print(f"\n{'=' * 80}")
